In [ ]:
#Cell 1
import os, warnings
import numpy as np
import pandas as pd
warnings.filterwarnings("ignore", category=RuntimeWarning)

SEED_DIGIT = 5      
ID_SUM     = 26      
LAST_NAME  = "Lancion"   
STUDENT_ID = "TUPM-26-1845"   
DATA_PATH   = "data/secom.data"
LABELS_PATH = "data/secom_labels.data"
MAX_MISSING_PCT = 50.0   
OUT_DIR = "ex1_measurement_quality"   
# ------------------------------------------------------
print("Student:", LAST_NAME, "| ID:", STUDENT_ID)
print("SEED_DIGIT =", SEED_DIGIT, "| ID_SUM =", ID_SUM)
X_df = pd.read_csv(DATA_PATH, sep=r"\s+", header=None).apply(pd.to_numeric, errors="coerce")
y = pd.read_csv(LABELS_PATH, sep=r"\s+", header=None, usecols=[0], names=["label"])["label"].to_numpy()
X_df.index.name = "record_id"          # original 0-based record identifier
X = X_df.to_numpy(dtype=float)
assert X.shape[0] == len(y), "data / label row mismatch"
fname = lambda i: f"F{i:03d}"

n_obs = (~np.isnan(X)).sum(axis=0)
missing_pct = np.isnan(X).mean(axis=0) * 100
constant = (n_obs == 0) | (np.nanmin(X, axis=0) == np.nanmax(X, axis=0))
usable_idx = np.where(~constant & (missing_pct <= MAX_MISSING_PCT))[0]
print("Matrix:", X.shape, "| Pass(-1):", (y == -1).sum(), "| Fail(1):", (y == 1).sum())
print("Features with missing:", (missing_pct > 0).sum(), "| constant/empty:", constant.sum(), "| usable:", len(usable_idx))
os.makedirs(OUT_DIR, exist_ok=True)

Student: Lancion | ID: TUPM-26-1845
SEED_DIGIT = 5 | ID_SUM = 26
Matrix: (1567, 590) | Pass(-1): 1463 | Fail(1): 104
Features with missing: 538 | constant/empty: 116 | usable: 446


In [7]:
# Cell 2
k = 5 + (SEED_DIGIT % 3)
rng1 = np.random.default_rng(SEED_DIGIT)
sel1 = np.sort(rng1.choice(usable_idx, size=k, replace=False))
Xs = X[:, sel1]
print("k =", k, "| Selected:", [fname(i) for i in sel1])


means   = np.nanmean(Xs, axis=0)
medians = np.nanmedian(Xs, axis=0)
stds    = np.nanstd(Xs, axis=0, ddof=1)          # sample SD
miss    = np.isnan(Xs).mean(axis=0) * 100
nobs    = (~np.isnan(Xs)).sum(axis=0)


mad = np.nanmedian(np.abs(Xs - medians), axis=0)
nz = mad > 0                                      # zero-MAD -> z = 0 (never flagged)
z = np.where(nz, 0.6745 * (Xs - medians) / np.where(nz, mad, 1.0), 0.0)
z[np.isnan(Xs)] = np.nan
print("Zero-MAD features:", [fname(i) for i in sel1[~nz]] or "None")


flagged  = np.abs(z) > 3.5                        # NaN compares False
flag_cnt = flagged.sum(axis=0)
flag_pct = flag_cnt / nobs * 100                  # % of observed values flagged
print("Flagged per feature:", dict(zip([fname(i) for i in sel1], flag_cnt)))


q = pd.DataFrame({
    "feature": [fname(i) for i in sel1], "column_index": sel1,
    "mean": means, "median": medians, "std_sample": stds,
    "missing_pct": miss, "mad": mad, "zero_mad": ~nz,
    "flagged_count": flag_cnt, "flagged_pct": flag_pct,
}).sort_values("flagged_pct", ascending=False).reset_index(drop=True)
q.insert(0, "rank", q.index + 1)
os.makedirs(OUT_DIR, exist_ok=True)
q.to_csv(os.path.join(OUT_DIR, "measurement_quality.csv"), index=False)
display(q)


print("Selected Features              :", list(q.sort_values("column_index")["feature"]), f"(k={k})")
print("Highest Missing Percentage     : %.4f%%" % miss.max(), "(", fname(sel1[miss.argmax()]), ")")
print("Feature w/ Highest Flagged %   :", q.loc[0, "feature"], "(%.4f%%)" % q.loc[0, "flagged_pct"])
print("Total Flagged Observations     :", int(flagged.sum()))
# Verification: per-column pandas cross-check of the vectorized flag counts
def pandas_flags(c):
    m = c.sub(c.median()).abs().median()
    return int(((0.6745 * (c - c.median()) / m).abs() > 3.5).sum()) if m > 0 else 0
chk = pd.DataFrame(Xs).apply(pandas_flags)
print("Vectorized Results Verified    :", bool(np.array_equal(chk.to_numpy(), flag_cnt)))

k = 7 | Selected: ['F011', 'F252', 'F286', 'F352', 'F368', 'F459', 'F467']
Zero-MAD features: None
Flagged per feature: {'F011': np.int64(5), 'F252': np.int64(5), 'F286': np.int64(21), 'F352': np.int64(40), 'F368': np.int64(56), 'F459': np.int64(3), 'F467': np.int64(55)}


,rank,feature,column_index,mean,median,std_sample,missing_pct,mad,zero_mad,flagged_count,flagged_pct
0,1,F368,368,0.003172,0.00280,0.002107,0.382897,0.0007,False,56,3.587444
1,2,F467,467,6.252091,5.64500,8.673724,0.255265,0.8195,False,55,3.518874
2,3,F352,352,0.023356,0.02200,0.013157,1.531589,0.0054,False,40,2.592353
3,4,F286,286,4.260018,3.36005,9.763829,0.191449,0.9200,False,21,1.342711
4,5,F011,11,0.964353,0.96580,0.012452,0.127632,0.0065,False,5,0.319489
5,6,F252,252,4.645115,2.86460,64.354756,0.000000,0.7668,False,5,0.319081
6,7,F459,459,2.838380,2.83035,1.345576,0.063816,0.5034,False,3,0.191571


Selected Features              : ['F011', 'F252', 'F286', 'F352', 'F368', 'F459', 'F467'] (k=7)
Highest Missing Percentage     : 1.5316% ( F352 )
Feature w/ Highest Flagged %   : F368 (3.5874%)
Total Flagged Observations     : 185
Vectorized Results Verified    : True
